# Подготовка

In [26]:
!python -V

Python 3.10.20


In [27]:
!where python

/Users/cyberdemon/Documents/practicum/rnn-lstm-gru/.conda/bin/python
/Users/cyberdemon/.pyenv/shims/python


In [28]:
import sys

print(sys.executable)

/Users/cyberdemon/Documents/practicum/rnn-lstm-gru/.conda/bin/python


In [29]:
%pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [30]:
%pip install -r requirements.txt

  Using cached scikit_learn-1.7.2-cp310-cp310-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached scipy-1.15.3-cp310-cp310-macosx_14_0_arm64.whl.metadata (61 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.7.2-cp310-cp310-macosx_12_0_arm64.whl (8.7 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached scipy-1.15.3-cp310-cp310-macosx_14_0_arm64.whl (22.4 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


# Загрузка и подготовка данных

In [31]:
from datasets import load_dataset

dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

print('Тип данных датасета:', type(dataset))

Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'wikitext-2-raw-v1' at /Users/cyberdemon/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Fri May  8 11:42:26 2026).


Тип данных датасета: <class 'datasets.dataset_dict.DatasetDict'>


In [32]:
from sklearn.model_selection import train_test_split

train_load = dataset['train']['text']
val_load = dataset['validation']['text']
test_load = dataset['test']['text']

# texts = dataset['train']['text']
# train_load, temp_texts = train_test_split(texts, test_size=0.2, random_state=42)
# val_load, test_load = train_test_split(temp_texts, test_size=0.5, random_state=42)

# train_load = train_load[:len(train_load) // 10]
# val_load = val_load[:len(val_load) // 10]
# test_load = test_load[:len(test_load) // 10]

print('Тренировочный датасет:', len(train_load))
print('Валидационный датасет:', len(val_load))
print('Тестовый датасет:', len(test_load))

Тренировочный датасет: 36718
Валидационный датасет: 3760
Тестовый датасет: 4358


In [33]:
import re

def preprocess_text(text: str) -> str:
    
    # приведение к нижнему регистру
    text = text.lower()

    # удаление всех символов, кроме букв, чисел и пробелов
    text = re.sub(r"[^a-zа-яё0-9 ]+", "", text)

    # удаление всех символов, кроме любых букв, чисел и пробелов
    # text = re.sub(r"[^\w ]|_", "", text)
    
    # удаление дублирующихся пробелов и пробелов в начале и в конце
    text = re.sub(r"\s+", " ", text).strip()

    return text

## Очищаем датасет

In [34]:
train_corpus = [preprocess_text(message) for message in train_load if message.strip()]
val_corpus = [preprocess_text(message) for message in val_load if message.strip()]
test_corpus = [preprocess_text(message) for message in test_load if message.strip()]

print('Первые 5 строк тренировочого датасета:')
for line in train_corpus[:5]:
    print(len(line), '\t', line)

Первые 5 строк тренировочого датасета:
23 	 valkyria chronicles iii
647 	 senj no valkyria 3 unrecorded chronicles japanese 3 lit valkyria of the battlefield 3 commonly referred to as valkyria chronicles iii outside japan is a tactical role playing video game developed by sega and mediavision for the playstation portable released in january 2011 in japan it is the third game in the valkyria series employing the same fusion of tactical and real time gameplay as its predecessors the story runs parallel to the first game and follows the nameless a penal military unit serving the nation of gallia during the second europan war who perform secret black operations and are pitted against the imperial unit calamaty raven
501 	 the game began development in 2010 carrying over a large portion of the work done on valkyria chronicles ii while it retained the standard features of the series it also underwent multiple adjustments such as making the game more forgiving for series newcomers character d

## Получаем списки слов из датасета

In [35]:
train_word = [message.split() for message in train_corpus]
val_word = [message.split() for message in val_corpus]
test_word = [message.split() for message in test_corpus]

print('Первые 5 списков слов тренировочого датасета:')
for line in train_word[:5]:
    print(len(line), '\t', line)

Первые 5 списков слов тренировочого датасета:
3 	 ['valkyria', 'chronicles', 'iii']
107 	 ['senj', 'no', 'valkyria', '3', 'unrecorded', 'chronicles', 'japanese', '3', 'lit', 'valkyria', 'of', 'the', 'battlefield', '3', 'commonly', 'referred', 'to', 'as', 'valkyria', 'chronicles', 'iii', 'outside', 'japan', 'is', 'a', 'tactical', 'role', 'playing', 'video', 'game', 'developed', 'by', 'sega', 'and', 'mediavision', 'for', 'the', 'playstation', 'portable', 'released', 'in', 'january', '2011', 'in', 'japan', 'it', 'is', 'the', 'third', 'game', 'in', 'the', 'valkyria', 'series', 'employing', 'the', 'same', 'fusion', 'of', 'tactical', 'and', 'real', 'time', 'gameplay', 'as', 'its', 'predecessors', 'the', 'story', 'runs', 'parallel', 'to', 'the', 'first', 'game', 'and', 'follows', 'the', 'nameless', 'a', 'penal', 'military', 'unit', 'serving', 'the', 'nation', 'of', 'gallia', 'during', 'the', 'second', 'europan', 'war', 'who', 'perform', 'secret', 'black', 'operations', 'and', 'are', 'pitted',

## Кастомный word-токенизатор

In [36]:
from collections import Counter

class WordTokenizer:
    def __init__(self, min_freq=8):

        self.min_freq = min_freq

        self.word2idx = {}
        self.idx2word = {}

        self.pad_token = "<PAD>"
        self.unk_token = "<UNK>"

    def fit(self, texts):

        counter = Counter()

        for words in texts:
            words = [word.lower() for word in words]
            counter.update(words)

        vocab = [self.pad_token, self.unk_token]

        for word, freq in counter.items():
            if freq >= self.min_freq:
                vocab.append(word)

        self.word2idx = {word: idx for idx, word in enumerate(vocab)}
        self.idx2word = {idx: word for word, idx in self.word2idx.items()}

    def encode(self, words, truncation = False, max_length = None):

        words = [word.lower() for word in words]

        tokens = []

        for word in words:
            token_id = self.word2idx.get(word, self.word2idx[self.unk_token])
            tokens.append(token_id)

        if max_length is not None and len(tokens) > max_length:
            if truncation:
                tokens = tokens[:max_length]
            else:
                raise ValueError(f"Слишком много токенов: {len(tokens)} > {max_length}")
        return tokens

    def decode(self, token_ids):

        words = []

        for token_id in token_ids:
            word = self.idx2word.get(token_id, self.unk_token)
            words.append(word)

        return words

    @property
    def vocab_size(self):
        return len(self.word2idx)

In [37]:
word_tokenizer = WordTokenizer(min_freq=5)

word_tokenizer.fit(train_word)

print("Vocabulary size:", word_tokenizer.vocab_size)

Vocabulary size: 20873


## Токенизировнный датасет

In [38]:
train_word_tokens = [word_tokenizer.encode(message) for message in train_word]
val_word_tokens = [word_tokenizer.encode(message) for message in val_word]
test_word_tokens = [word_tokenizer.encode(message) for message in test_word]

## Пример токенизации и декодирования на простой строке

In [39]:
text = "the cat sat randomword on the mat".split()
print('Исходный текст:\n', text)

tokens = word_tokenizer.encode(text)
print('Токенизированный текст:\n', tokens)

decoded = word_tokenizer.decode(tokens)
print('Декодированный текст:\n', decoded)

Исходный текст:
 ['the', 'cat', 'sat', 'randomword', 'on', 'the', 'mat']
Токенизированный текст:
 [11, 4085, 4107, 1, 82, 11, 18400]
Декодированный текст:
 ['the', 'cat', 'sat', '<UNK>', 'on', 'the', 'mat']


## Train

In [40]:
print('#ORIGINAL')
for message in train_load[:8]:
    if not message:
        continue
    print(len(message), '\t', message, end="")

print('\n#LOWKEY, NO SYMBOL')
for message in train_corpus[:5]:
    print(len(message), '\t', message)

print('\n#WORD')
for word in train_word[:3]:
    print(word)

print('\n#WORD (ENCODED)')
for tokens in train_word_tokens[:3]:
    print(tokens)

print('\n#WORD (DECODED)')
for tokens in train_word_tokens[:3]:
    print(word_tokenizer.decode(tokens))

#ORIGINAL
30 	  = Valkyria Chronicles III = 
706 	  Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . 
524 	  The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game mor

## Validation

In [41]:
print('#ORIGINAL')
for message in val_load[:9]:
    if not message:
        continue
    print(len(message), '\t', message, end="")

print('\n#LOWKEY, NO SYMBOL')
for message in val_corpus[:5]:
    print(len(message), '\t', message)

print('\n#WORD')
for word in val_word[:3]:
    print(word)

print('\n#WORD (TOKENIZED)')
for tokens in val_word_tokens[:3]:
    print(tokens)

print('\n#WORD (DECODED)')
for tokens in val_word_tokens[:3]:
    print(word_tokenizer.decode(tokens))

#ORIGINAL
23 	  = Homarus gammarus = 
702 	  Homarus gammarus , known as the European lobster or common lobster , is a species of clawed lobster from the eastern Atlantic Ocean , Mediterranean Sea and parts of the Black Sea . It is closely related to the American lobster , H. americanus . It may grow to a length of 60 cm ( 24 in ) and a mass of 6 kilograms ( 13 lb ) , and bears a conspicuous pair of claws . In life , the lobsters are blue , only becoming " lobster red " on cooking . Mating occurs in the summer , producing eggs which are carried by the females for up to a year before hatching into planktonic larvae . Homarus gammarus is a highly esteemed food , and is widely caught using lobster pots , mostly around the British Isles . 
22 	  = = Description = = 
549 	  Homarus gammarus is a large crustacean , with a body length up to 60 centimetres ( 24 in ) and weighing up to 5 – 6 kilograms ( 11 – 13 lb ) , although the lobsters caught in lobster pots are usually 23 – 38 cm ( 9 – 15 

## Test

In [42]:
print('#ORIGINAL')
for message in test_load[:10]:
    if not message:
        continue
    print(len(message), '\t', message, end="")

print('\n#LOWKEY, NO SYMBOL')
for message in test_corpus[:5]:
    print(len(message), '\t', message)

print('\n#WORD')
for word in test_word[:3]:
    print(word)

print('\n#WORD (TOKENIZED)')
for tokens in test_word_tokens[:3]:
    print(tokens)

print('\n#WORD (DECODED)')
for tokens in test_word_tokens[:3]:
    print(word_tokenizer.decode(tokens))

#ORIGINAL
21 	  = Robert Boulter = 
859 	  Robert Boulter is an English film , television and theatre actor . He had a guest @-@ starring role on the television series The Bill in 2000 . This was followed by a starring role in the play Herons written by Simon Stephens , which was performed in 2001 at the Royal Court Theatre . He had a guest role in the television series Judge John Deed in 2002 . In 2004 Boulter landed a role as " Craig " in the episode " Teddy 's Story " of the television series The Long Firm ; he starred alongside actors Mark Strong and Derek Jacobi . He was cast in the 2005 theatre productions of the Philip Ridley play Mercury Fur , which was performed at the Drum Theatre in Plymouth and the Menier Chocolate Factory in London . He was directed by John Tiffany and starred alongside Ben Whishaw , Shane Zaza , Harry Kent , Fraser Ayres , Sophie Stanton and Dominic Hall . 
861 	  In 2006 , Boulter starred alongside Whishaw in the play Citizenship written by Mark Ravenhil